# Lab - Prompt Caching: Pay for the Policy Once

Three tasks on sending the same long document with every request and paying for it once. ShopAssist answers every question from the same return policy; without caching, the policy is billed in full on every call.

| Task | What you learn |
| --- | --- |
| 1 | Two system blocks, the policy marked with cache_control - read usage |
| 2 | Write once, read after: price each request in code |
| 3 | A changing value before the mark breaks every match - move it after |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell runs offline - including the prompt cache: prefixes, the cache_control mark, the minimum length and the usage numbers behave as the API's do. The code you write is exactly the code you would write against the real API.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import uuid
from pathlib import Path

load_dotenv()
client = Anthropic()
shopassist_lab.reset()   # offline only: start with an empty prompt cache

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
POLICY = Path("policy.txt").read_text()   # sixteen sections, the same for every question
INSTRUCTION = "You are ShopAssist. Answer from the policy below in one sentence."
QUESTIONS = ["Can I return earrings?",
             "How long does a card refund take?",
             "Who approves a refund of 250?"]
print(len(POLICY), "characters of policy")

---

## Task 1 - Mark the stable prefix

In [ ]:
# ============================================================
# TASK 1 - Mark the stable prefix
# ============================================================
#
# WHAT TO DO
#   Make the system prompt a list of two blocks: the short
#   instruction, then the policy with cache_control - the mark.
#   The question goes after, in messages. Send the first
#   question and read the three numbers in usage.
#
# WHY IT MATTERS
#   Every question is answered from the same long policy, and
#   without caching it is billed in full on every call. Prompt
#   caching works on prefixes: the API remembers the beginning
#   of a request up to the mark, and the next request that
#   starts the same way reuses it. Everything before the mark is
#   cached; the question, after it, is not.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The system prompt becomes a list of two blocks".
#
# HOW TO DO IT
#   Three blanks:
#
#       POLICY                  the long, stable text
#       {"type": "ephemeral"}   the mark: cache everything up to here
#       question                the changing part, after the mark
# ============================================================

# TODO: replace each ... below with the value named beside it
def ask(question, system=None):
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        output_config={"effort": "low"},
        system=system or [
            {"type": "text", "text": INSTRUCTION},
            {"type": "text", "text": ..., "cache_control": ...},   # POLICY, {"type": "ephemeral"}
        ],
        messages=[{"role": "user", "content": ...}],   # question
    )
    u = response.usage
    print(f"input {u.input_tokens:5} | cache write {u.cache_creation_input_tokens:5} | "
          f"cache read {u.cache_read_input_tokens:5} | {response.content[0].text[:60]}")
    return response

first = ask(QUESTIONS[0])

check("cache_mark", first=first)

---

## Task 2 - Read the cache, price the difference

In [ ]:
# ============================================================
# TASK 2 - Read the cache, price the difference
# ============================================================
#
# WHAT TO DO
#   Ask all three questions and keep each request's usage as a
#   row. Write cost(): the price of one row in dollars, at
#   Sonnet 5.5's prices. Compare with what the same three
#   requests would cost without the cache.
#
# WHY IT MATTERS
#   Writing the cache costs a quarter more than normal input,
#   once; reading it costs a tenth. The first request writes,
#   the next ones read - and only the question is billed as
#   normal input. Usage is how you know caching happened; a
#   cost function turns it into the number someone will ask you
#   for.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Question one: input fourteen, cache write about nineteen
#   hundred".
#
# HOW TO DO IT
#   Three blanks:
#
#       2.50   dollars per million tokens written to the cache
#       0.20   dollars per million tokens read from it
#       rows   every request's usage
# ============================================================

PRICE = {"input": 2.00, "output": 10.00}   # provided: dollars per million tokens

shopassist_lab.reset()   # provided: an empty cache, so the first question writes it
try:
    # TODO: replace each ... below with the value named beside it
    def cost(row):
        return (row["input"] * PRICE["input"] + row["cache_write"] * ...   # 2.50
                + row["cache_read"] * ... + row["output"] * PRICE["output"]) / 1_000_000   # 0.20
    
    rows = []
    for question in QUESTIONS:
        u = ask(question).usage
        rows.append({"input": u.input_tokens, "cache_write": u.cache_creation_input_tokens,
                     "cache_read": u.cache_read_input_tokens, "output": u.output_tokens})
    cached = sum(cost(r) for r in ...)   # rows
    uncached = sum(cost({"input": r["input"] + r["cache_write"] + r["cache_read"],
                         "cache_write": 0, "cache_read": 0, "output": r["output"]})
                   for r in rows)
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(f"three questions: ${cached:.5f} with the cache, ${uncached:.5f} without")

check("cache_reads", rows=rows, cost=cost)

---

## Task 3 - Stable first, changing after the mark

In [ ]:
# ============================================================
# TASK 3 - Stable first, changing after the mark
# ============================================================
#
# WHAT TO DO
#   A teammate adds a request id for tracing - inside the
#   instruction block, before the policy. Run the questions and
#   watch the cache. Then fix it: keep the system blocks
#   identical, and send the id after the mark, with the
#   question.
#
# WHY IT MATTERS
#   Caching is a prefix match: change one byte before the mark
#   and nothing matches - every call writes a new cache and
#   pays full price, with no error to warn you. A timestamp, a
#   request id, a shuffled tool list are the usual culprits.
#   The rule: stable content first, with the mark; changing
#   content after it - and check that cache reads carry the big
#   number.
#
# WHERE TO SEE IT IN THE LECTURE
#   "It is tempting to put the question before the policy".
#
# HOW TO DO IT
#   Three blanks:
#
#       INSTRUCTION    the stable instruction, without the id
#       request_id     the id, now part of the question
#       fixed          the rows of the fixed version
# ============================================================


def row_of(response):   # provided
    u = response.usage
    return {"cache_write": u.cache_creation_input_tokens, "cache_read": u.cache_read_input_tokens}


print("broken - a request id before the mark:")
broken = []
for question in QUESTIONS:
    request_id = uuid.uuid4().hex[:8]
    system = [{"type": "text", "text": f"Request {request_id}. " + INSTRUCTION},
              {"type": "text", "text": POLICY, "cache_control": {"type": "ephemeral"}}]
    broken.append(row_of(ask(question, system)))

print("fixed - the id after the mark:")
try:
    # TODO: replace each ... below with the value named beside it
    fixed = []
    for question in QUESTIONS:
        request_id = uuid.uuid4().hex[:8]
        system = [{"type": "text", "text": ...},   # INSTRUCTION - nothing that changes
                  {"type": "text", "text": POLICY, "cache_control": {"type": "ephemeral"}}]
        ....append(row_of(ask(f"[request {...}] {question}", system)))   # fixed, request_id
except (TypeError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("stable_prefix", broken=broken, fixed=fixed)

---

## Done

Put what never changes at the front, mark it with cache_control, keep the changing part after it, and check usage: cache read should carry the big number, and input only the question. Price it in code - and watch for anything that changes the prefix.